# coconut-audit — HTML report XSS safety

Demonstrates that every user-controlled field flowing into `render_html_report`
is HTML-escaped via `html.escape(..., quote=True)`. This inherits the
`recurrentlens 0.1.0.post1` XSS-hotfix lesson, with the policy moved to the
Python stdlib so each field's escape call is grep-auditable.

If you ever ship a report with an unescaped field, this notebook will fail.

In [ ]:
%pip install -q coconut-audit

In [ ]:
from pathlib import Path

from coconut_audit.core.types import AuditReport, AuditVerdict, ProbeKind
from coconut_audit.reports import render_html_report

payload = "<script>alert('xss')</script>"
report = AuditReport(
    audit_id=payload,
    model_id=payload,
    sae_id=payload,
    probe_kind=ProbeKind.STEERING,
    verdict=AuditVerdict.WARN,
    metrics={f'attr="{payload}"': 1.0},
    notes=[payload],
)

out = render_html_report(report, Path('xss_demo.html'))
html = out.read_text('utf-8')

assert '<script>' not in html, 'raw <script> leaked — XSS regression!'
assert '</script>' not in html, 'raw </script> leaked — XSS regression!'
assert '&lt;script&gt;' in html, 'expected escaped <script> in output'
print('XSS escape verified across model_id / sae_id / audit_id / notes / metric keys.')
print(f'See {out} (rendered safely).')